In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import scanpy as sc
import pandas as pd
import scanpy.external as sce
import scvi

In [ ]:
adata = sc.read_h5ad(f"{ACC_DATA_ROOT}/ouptuts/concatenated_outputs/raw_merged_counts_good.h5ad")

In [ ]:
# normalization
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    batch_key="batch",
    flavor="seurat_v3",
    n_top_genes=3000
)

In [ ]:
# subset to only contain the HVG
adata = adata[:, adata.var.highly_variable].copy()

In [ ]:
from scvi.model import SCVI
# from scvi.data import setup_anndata

In [ ]:
# 4B  scVI --------------------------------------------------------------------
adata_vi = adata.copy()                                  # still has log-norm X
SCVI.setup_anndata(
    adata_vi,
    batch_key="batch",
    layer="counts"      # raw counts
)
vae = scvi.model.SCVI(adata_vi, n_latent=30)
vae.train(max_epochs=300, check_val_every_n_epoch=10)

adata_vi.obsm["X_scVI"] = vae.get_latent_representation()
sc.pp.neighbors(adata_vi, use_rep="X_scVI")
sc.tl.umap(adata_vi, min_dist=0.3)
sc.tl.leiden(adata_vi, resolution=0.6, key_added="leiden_vi")

In [ ]:
adata_vi

In [ ]:
sc.pl.umap(adata_vi, color = ['leiden_vi'], size = 5)

In [ ]:
sc.pl.umap(adata_vi, color = ['batch'], size = 5)